# Volatility Dynamics 10c — long-history physical tail for 0DTE wings

**VD 023.** Module 10b found the wings RICH against realized outcomes, but on only 18 sessions. That leaves a peso problem: one ±8σ day per asset-year would erase the bid-side premium. 10c replaces the outcome pool with about two years of 5-minute history, strictly before the first quote, so the pool is fully out of sample.

**Source.** Binance spot `<ASSET>USDT` closes (index constituents), with the Deribit perpetual as fallback, downloaded by `tools/download_price_history.py`.

**Outcomes.** Built exactly as in 10b:

$$u=\ln(S_T/I_t)/\sqrt{D_t}$$

using the Module 08 clock, backward 4h bipower and settlement weights.

**Clock.** $D$ uses exactly Module 10's clock, so pool outcomes and quotes are standardized identically. A clock refitted on the long history is reported as a sensitivity check.

**Pre-registered (VD 023):**
- **Gate.** On the quote window, the long-history $u$ is compared with the index-based $u$ from 10b, per asset. An asset passes if the correlation is at least 0.90 and the sd ratio lies in [0.85, 1.15]. Failing assets are removed from the pool; if more than 2 assets fail, the verdict is GATE_FAIL.
- **Decision V_bid.** $M$ = median of the session-median $\ln(\text{bid}/P)$ on Module 10's primary rows. The CI comes from a joint bootstrap that resamples quote sessions and history dates (B = 1000), giving RICH, CHEAP or CONSISTENT. **V_mid** is computed the same way for continuity with 10 and 10b.
- **Consequence.** If V_bid is RICH, the next step is a pre-registered, defined-risk wing-selling backtest. Otherwise no wing-selling research from this branch until there are more sessions.

Reported only: net-of-fee bid, per asset, buckets, TTE-conditioned and own-asset pools, tails by volatility tercile, and break-even disaster mass. Outputs: `10c/`.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_jump_option_wings as ow
from data_engineering.modeling import volatility_wings_empirical as we
from data_engineering.modeling import volatility_wings_longhistory as wl
ASSETS = [x.strip().upper() for x in os.getenv('VD10C_ASSETS', 'AVAX,BTC,ETH,HYPE,SOL,TRX,XRP').split(',') if x.strip()]
HIST_DIR = Path(os.getenv('VD10C_HISTORY_DIR', str(DATA_ROOT / 'history' / 'price_5m')))
N_BOOT = int(os.getenv('VD10C_N_BOOT', str(wl.N_BOOT)))
VD = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics'
OUT = VD / '10c'
OUT.mkdir(parents=True, exist_ok=True)
print('Module 10c', wl.VERSION, wl.selftest())
manifest = HIST_DIR / 'manifest.json'
if manifest.exists():
    print(manifest.read_text())


In [ ]:
t0 = time.time()
hist = wl.load_history(HIST_DIR, ASSETS)
print(hist.groupby('underlying').agg(first=('state_time', 'min'), last=('state_time', 'max'), n=('close', 'size'),
                                     source=('source', lambda s: ','.join(sorted(set(s))))))
frame = wl.return_frame(hist)
from data_engineering.modeling import volatility_vd_b_core as vb
con = duckdb.connect(str(DB_PATH), read_only=True)
dbf = con.execute("""SELECT state_time, underlying, return_5m FROM time_series_feature_state
                     WHERE return_5m IS NOT NULL""").df()
con.close()
dbf = dbf[dbf.underlying.isin(ASSETS) & (pd.to_datetime(dbf.state_time, utc=True) < ow.VRP_CONFIRMATION_START)]
clock = wl.module10_clock(vb.build_grid(dbf))
oc = wl.long_outcomes(frame, clock=clock)
oc_refit = wl.long_outcomes(frame)
print(f'{len(oc):,} long-history origins in {time.time()-t0:.0f}s')
q = pd.read_csv(VD / '10' / '10_priced_quotes.csv.gz')
o10b = pd.read_csv(VD / '10b' / '10b_origin_outcomes.csv.gz')
pool_all, overlap = wl.split(oc, sorted(q.session.unique()))
gate = wl.consistency_gate(overlap, o10b)
display(gate.round(3))
pool_refit, overlap_refit = wl.split(oc_refit, sorted(q.session.unique()))
gate_refit = wl.consistency_gate(overlap_refit, o10b)
print('sensitivity: clock refitted on long history')
display(gate_refit.round(3))
failed = gate.loc[~gate['pass'], 'underlying'].tolist()
gate_fail = len(failed) > wl.GATE_MAX_FAILS
pool = pool_all[~pool_all.underlying.isin(failed)].reset_index(drop=True)
print('gate failures:', failed, '| GATE_FAIL:', gate_fail)
print(f'pool: {len(pool):,} origins, {pool.session.nunique()} dates, {pool.session.min()} -> {pool.session.max()}')


In [ ]:
excluded = json.loads((VD / '10' / '10_decision.json').read_text()).get('excluded_assets_tick_artifact', [])
prim = ow.primary_sample(q, excluded).reset_index(drop=True)
prim['net_bid_usd'] = wl.net_bid(prim)
t0 = time.time()
dec = wl.decide(prim, pool, n_boot=N_BOOT, price_cols=('bid_usd', 'mid_usd', 'net_bid_usd'), gate_fail=gate_fail)
print(f'joint bootstrap ({time.time()-t0:.0f}s)')
print(json.dumps(dec, indent=2, default=float))


In [ ]:
P = wl.price(prim, pool.u.to_numpy())
by_asset = pd.DataFrame([{'underlying': u, 'price': c, **wl.quote_only(g, P[g.index], c)}
                         for u, g in prim.groupby('underlying') for c in ('bid_usd', 'mid_usd')])
display(by_asset.round(3))
own = []
for u, g in prim.groupby('underlying'):
    pu = pool[pool.underlying == u]
    if len(pu) < 1000:
        continue
    Pu = wl.price(g, pu.u.to_numpy())
    own.append({'underlying': u, **wl.quote_only(g, Pu, 'bid_usd')})
own = pd.DataFrame(own)
display(own.round(3))
tte = []
for tb, g in prim.groupby('tte_bucket'):
    pt = pool[pool.tte_bucket == tb]
    Pt = wl.price(g, pt.u.to_numpy())
    tte.append({'tte_bucket': tb, **{f'{c}_{k}': v for c in ('bid_usd', 'mid_usd')
                                     for k, v in wl.quote_only(g, Pt, c).items() if k in ('M', 'ci_low', 'ci_high', 'verdict', 'n_rows')}})
tte = pd.DataFrame(tte)
display(tte.round(3))


In [ ]:
allq = q[(~q.floor_bound) & (~q.underlying.isin(excluded))].reset_index(drop=True)
Pa = wl.price(allq, pool.u.to_numpy())
rows = []
for (zb, tb), g in allq.groupby(['z_bucket', 'tte_bucket']):
    for c in ('bid_usd', 'mid_usd'):
        r = wl.quote_only(g, Pa[g.index], c, n_boot=500)
        rows.append({'z_bucket': zb, 'tte_bucket': tb, 'price': c, **{k: r.get(k) for k in ('n_rows', 'n_sessions', 'M', 'ci_low', 'ci_high', 'verdict', 'share_model_price_zero')}})
buckets = pd.DataFrame(rows)
display(buckets[buckets.price == 'bid_usd'].pivot(index='z_bucket', columns='tte_bucket', values='M').round(3))
tails = we.tail_table(pool)
display(tails.round(4))
vt = wl.vol_tercile_tails(pool)
display(vt.round(4))
be = pd.DataFrame([{'u_star': us, 'price': c, 'breakeven_p_per_origin': wl.breakeven(prim, pool.u.to_numpy(), c, us),
                    'observed_share_beyond': float(np.mean(np.abs(pool.u) > us))}
                   for us in wl.BREAKEVEN_USTAR for c in ('bid_usd', 'mid_usd')])
display(be)
pr = pool_refit[~pool_refit.underlying.isin(failed)]
refit_sens = {c: wl.quote_only(prim, wl.price(prim, pr.u.to_numpy()), c) for c in ('bid_usd', 'mid_usd')}
print('sensitivity, refitted clock pool:', json.dumps(refit_sens, indent=2, default=float))


In [ ]:
gate.to_csv(OUT / '10c_consistency_gate.csv', index=False)
gate_refit.to_csv(OUT / '10c_consistency_gate_refit_clock.csv', index=False)
pd.DataFrame([{'price': c, **v} for c, v in refit_sens.items()]).to_csv(OUT / '10c_refit_clock_sensitivity.csv', index=False)
by_asset.to_csv(OUT / '10c_by_asset.csv', index=False)
own.to_csv(OUT / '10c_own_asset_pool_bid.csv', index=False)
tte.to_csv(OUT / '10c_tte_conditioned.csv', index=False)
buckets.to_csv(OUT / '10c_bucket_table.csv', index=False)
tails.to_csv(OUT / '10c_tail_frequencies.csv', index=False)
vt.to_csv(OUT / '10c_tails_by_vol_tercile.csv', index=False)
be.to_csv(OUT / '10c_breakeven_disaster.csv', index=False)
if os.getenv('VD10C_EXPORT_POOL', '0') == '1':      # ~50 MB; off by default to keep the repo small
    pool.drop(columns=['bar_index']).to_csv(OUT / '10c_pool_outcomes.csv.gz', index=False, compression='gzip')
pool.groupby(['underlying', 'session']).u.agg(['size', 'min', 'max', 'std']).reset_index().to_csv(
    OUT / '10c_pool_session_summary.csv.gz', index=False, compression='gzip')
decision = {'V_bid': dec['bid_usd'], 'V_mid': dec['mid_usd'], 'net_bid_reported': dec['net_bid_usd'],
            'gate_failures': failed, 'gate_fail': gate_fail, 'pool': {k: dec[k] for k in ('n_pool_origins', 'n_pool_dates')},
            'share_model_price_zero': dec['share_model_price_zero']}
(OUT / '10c_decision.json').write_text(json.dumps(decision, indent=2, default=float))
meta = {'version': wl.VERSION, 'module': '10c_long_history_wing_benchmark', 'history_dir': str(HIST_DIR),
        'sources': sorted(set(hist.source)), 'pool_last_session': wl.POOL_LAST_SESSION, 'n_boot': N_BOOT,
        'tail_raw_share': wl.TAIL_RAW_LONG, 'consumes': ['10/10_priced_quotes.csv.gz', '10b/10b_origin_outcomes.csv.gz']}
(OUT / '10c_run_metadata.json').write_text(json.dumps(meta, indent=2, default=float))
print(json.dumps(decision, indent=2, default=float))
